<a href="https://colab.research.google.com/github/AbdulWaheed-21/Machine-Learning-Internship-FlyRank/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulWaheed-21/Machine-Learning-Internship-FlyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

My lane is Refresh / Content Opportunity Scoring, one of the core lanes named in the dataset and lane guide. FlyRank pages age, and their organic search performance drifts over time, so someone on the content team has to decide which pages get reviewed and rewritten first with a limited number of writing hours each month. I picked this lane because the raw material for a real decision already sits inside the anonymized dataset, search volume, current position, click through rate, and how long since a page was last touched. The sample also carries thirty two client accounts with a median of about five hundred sixty seven pages each, which is far too many for any single editor to review by eye every month. I want to test whether a ranked list built from these signals actually points at the pages worth a writer's time first, rather than at pages that look busy but would never move. I am also naming the starter label's limit honestly from the start. The starter pipeline marks a page declining when trend_direction reads down, which is a bucket calculated from the current window, not a real future outcome, so it is a beginner proxy rather than the final target. A stronger version to build toward in a later week would use prior ninety days of features to predict decline over the next thirty days, and I plan to treat this week's label as a starting point to test against, not as the finished definition.

In [2]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("rows and columns:", df.shape)
print("unique clients:", df["client_id"].nunique())
print("median pages per client:", int(df.groupby("client_id").size().median()))

rows and columns: (30000, 44)
unique clients: 32
median pages per client: 567


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The decision this work should improve is short to state and hard to do well: given a list of published pages, which ones should a content writer refresh this month, and in what order. The people who act on the recommendation are a content strategist and the account manager working with that client, who together usually have room for somewhere between five and twenty page refreshes a month, not five hundred. A wrong call carries two separate costs. If the ranked list sends a writer to refresh a page that was never going to move, the client pays for hours that produced nothing, and a page that truly needed help stayed untouched instead. If the ranked list misses a page that is quietly losing position and clicks, the client keeps losing real traffic and revenue for weeks before a human notices, since decline across thousands of pages is slow and easy to miss without a system watching for it.

In [4]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# a rough monthly writing budget check: how many pages does a typical client carry
pages_per_client = df.groupby("client_id").size()
print("smallest client page count:", pages_per_client.min())
print("largest client page count:", pages_per_client.max())
print("a five to twenty page monthly budget covers a tiny slice of that, so order matters")


smallest client page count: 3
largest client page count: 7008
a five to twenty page monthly budget covers a tiny slice of that, so order matters


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the anonymized sample make this lane worth the next seven weeks. Out of thirty thousand pages, eight thousand seven hundred thirteen of them, about twenty nine percent, already have meaningful traffic, more than one hundred impressions in the last thirty days, and a downward trend. That declining group carries a click through rate of about zero point two five percent, roughly half the site wide average of zero point five one percent. The same declining group also waited far longer for attention, about fifty four days since the last content update on average, against a twenty day median across the full sample. Together these numbers describe a large and specific set of pages that are actively losing ground while sitting untouched, which is exactly the pattern a prioritization system is meant to catch early instead of by accident.

In [6]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

declining = (df["impressions_last_30d"] > 100) & (df["trend_direction"] == "down")

share_declining = declining.mean() * 100
ctr_declining = df.loc[declining, "ctr"].mean()
ctr_overall = df["ctr"].mean()
days_declining = df.loc[declining, "days_since_last_update"].mean()
days_overall_median = df["days_since_last_update"].median()

print(f"pages with real traffic and a downward trend: {declining.sum()} of {len(df)} ({share_declining:.1f} percent)")
print(f"average click through rate, declining group: {ctr_declining:.2f} percent")
print(f"average click through rate, whole sample: {ctr_overall:.2f} percent")
print(f"average days since last update, declining group: {days_declining:.1f}")
print(f"median days since last update, whole sample: {days_overall_median:.1f}")

pages with real traffic and a downward trend: 8713 of 30000 (29.0 percent)
average click through rate, declining group: 0.25 percent
average click through rate, whole sample: 0.51 percent
average days since last update, declining group: 53.9
median days since last update, whole sample: 20.0


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This work can say what is observed and what is directional, nothing stronger. It can say that pages sharing certain measurable traits in this sample, page age, time since last update, and click through rate relative to position, are associated with a higher chance of continued decline. It can offer a ranked queue as decision support, a starting point for a human editor to review and question, not a final verdict handed down by a machine. It cannot say why Google ranks any single page the way it does, and it cannot say that refreshing a page will cause its ranking to improve, because the data here shows patterns across many pages and clients, not a controlled experiment run on any one page. Every claim I write for this project will stay inside the range of observed, measured, directional, and decision support, and will avoid language like predicting Google or proving that a refresh caused a result.

In [7]:

careful_words = ["observed", "measured", "directional", "decision support"]
avoid_words = ["predicts Google", "proves", "causes", "guarantees"]

print("words I can use:", careful_words)
print("words I will not use without a controlled test:", avoid_words)


words I can use: ['observed', 'measured', 'directional', 'decision support']
words I will not use without a controlled test: ['predicts Google', 'proves', 'causes', 'guarantees']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.